In [ ]:
import sys
!{sys.executable} -m pip install easyocr pymupdf --trusted-host pypi.org --trusted-host files.pythonhosted.org

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade pip

In [ ]:
import sys
!{sys.executable} -m pip install --ignore-installed scipy==1.10.1

In [ ]:
import sys
!{sys.executable} -m pip install pymupdf langchain-text-splitters

In [ ]:
import os
import re
import pymupdf as fitz
import easyocr
import numpy as np
from PIL import Image
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Загрузка OCR-модели...")
reader = easyocr.Reader(['ru', 'en'], gpu=False) 

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       
    chunk_overlap=150,    
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""] 
)

def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s\.\,\;\:\!\?\-\«\»\(\)\%\+\/\d]', '', text)
    return text.strip()

def extract_text_from_pdf_smart(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""
    
    for page_num, page in enumerate(doc):
        page_text = page.get_text()
        
        if len(page_text.strip()) < 50:
            print(f"  [OCR] Страница {page_num + 1} (скан)...")
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            result = reader.readtext(np.array(img), detail=0, paragraph=True)
            page_text = "\n".join(result)
            
        full_text += page_text + "\n"
        
    return full_text

def process_pdf_to_chunks(pdf_path):
    file_name = os.path.basename(pdf_path)
    print(f"Обработка файла: {file_name}")
    
    raw_text = extract_text_from_pdf_smart(pdf_path)
    clean_txt = clean_text(raw_text)
    
    if len(clean_txt) < 100:
        print(f"Файл {file_name} пуст или не распознан. Пропускаем.")
        return []
        
    chunks = text_splitter.split_text(clean_txt)
    print(f"Получено {len(chunks)} чанков (длина текста: {len(clean_txt)})")
    
    return [{"text": chunk, "source": file_name} for chunk in chunks]

def build_knowledge_base(pdf_folder_path):
    all_data = []
    
    for filename in os.listdir(pdf_folder_path):
        if filename.lower().endswith('.pdf'):
            file_path = os.path.join(pdf_folder_path, filename)
            chunks = process_pdf_to_chunks(file_path)
            all_data.extend(chunks)
            
    print(f"\nВСЕГО СОБРАНО ЧАНКОВ: {len(all_data)}")
    return all_data

PDF_FOLDER = "../data/pdf/" 

all_data = build_knowledge_base(PDF_FOLDER)

if all_data:
    print(f"\nПример первого чанка:\n{all_data[0]['source']} \n{all_data[0]['text'][:300]}...")

In [ ]:
import json
import os

def save_knowledge_base(all_data, filepath="knowledge_base.json"):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(all_data, f, ensure_ascii=False, indent=2)
    print(f"База сохранена в {filepath} ({len(all_data)} чанков)")

def load_knowledge_base(filepath="knowledge_base.json"):
    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"База загружена из {filepath} ({len(data)} чанков)")
        return data
    else:
        print(f"Файл {filepath} не найден. Возвращаю пустую базу.")
        return []

save_knowledge_base(all_data)

In [ ]:
def add_document(file_path, kb_filepath="knowledge_base.json"):
    file_name = os.path.basename(file_path)
    
    current_data = load_knowledge_base(kb_filepath)
    
    current_data = [c for c in current_data if c["source"] != file_name]
    
    new_chunks = process_pdf_to_chunks(file_path)
    
    if not new_chunks:
        print(f"Не удалось извлечь текст из {file_name}")
        return {"status": "error", "chunks_added": 0}
    
    current_data.extend(new_chunks)
    
    save_knowledge_base(current_data, kb_filepath)
    
    return {"status": "success", "chunks_added": len(new_chunks), "source": file_name}


def delete_document(file_name, kb_filepath="knowledge_base.json"):

    current_data = load_knowledge_base(kb_filepath)
    
    filtered_data = [c for c in current_data if c["source"] != file_name]
    
    deleted_count = len(current_data) - len(filtered_data)
    
    if deleted_count == 0:
        print(f"Файл {file_name} не найден в базе.")
        return {"status": "not_found", "chunks_deleted": 0}
    
    save_knowledge_base(filtered_data, kb_filepath)
    
    return {"status": "success", "chunks_deleted": deleted_count, "source": file_name}